# Reproducing Huszár et al. (2022) from DANDI:000552

Paper. Huszár R, Zhang Y, Blockus H, Buzsáki G (2022). Preconfigured dynamics in the hippocampus are
guided by embryonic birthdate and rate of neurogenesis. *Nature Neuroscience* 25, 1201-1212.
doi:10.1038/s41593-022-01138-x.

Claims. `claims/huszar-2022.yaml`, written from the author manuscript in PubMed Central (PMC10807234)
with its Reporting Summary and Supplementary Information (Supplementary Tables 1 to 5), and the paper's code
(github.com/rhuszar/optotagCA1_final, commit 51320d4). Method details the claims file lacks are taken from
the Methods and legends of that text, cited beside the code in `huszar.py`.

Dataset. DANDI:000552, version `0.261113.1035` (doi:10.48324/dandi.000552/0.261113.1035), the most recent
published version. The paper cites no version (it was published before the dandiset). The earlier published
version, 0.230630.2304, lacks what the claims need: its release notes say it had no light-responsive labels,
no light pulses, units for only some sessions and LFP in only 13 files, while 0.261113.1035 holds all of
these for the 104 sessions. It therefore also matches the claims file's description better. Only the 104
processed files are read, streamed from the archive; the 11 raw wide-band files are not.

This notebook streams every processed file and takes about 35 minutes, most of it reading one LFP channel
per session for the theta analyses.

| Claim | Tested here |
|---|---|
| cohort-birthdates, st1-animal-sessions, sessions-durations, spwr-duration, unit-count, ed2-light-responsive-count | yes |
| fig1e-labelled-fraction, fig1f-rem-shift, fig1g-rate-by-birthdate, fig1h-burst-by-birthdate | yes |
| fig2b-synchrony-per-animal, fig2c-theta-cofiring, fig2d-theta-depth, fig2f-ripple-firing-by-birthdate, fig2g-ripple-cofiring | yes |
| maze-cohort, maze-task, novel-explorations | yes |
| fig5b-spatial-info-by-birthdate, fig5c-sbd-ratemap-overlap, fig5d-overlap-by-distance, fig5e-pv-decorrelation, fig5f-remap-together | yes |
| fig6c-theta-rate-preserved, fig6d-rate-lower-novel, fig6d-info-lower-novel, fig6c-novel-ratemap-corr-higher, fig6e-sbd-novel-overlap, fig6f-remap-together-novel | yes, open fields left out |
| fig7b-sbd-convergence, fig7ce-convergence-by-birthdate, fig7d-transmission-gamma, fig7fg-convergence-cofiring | yes |
| ed1-brdu-specificity, fig1c-radial-depth | blocked: histology not in the dataset |
| ed2-waveform-validation | blocked: light-evoked waveforms need the wide-band signal, absent for 91 of 96 sessions |
| ed5-matched-controls | blocked: no L-ratio in the dataset |
| fig3c, fig3e, ed6, ed7 (assemblies), fig4d-model-fit | blocked: too large for this run |
| ed3-depth-control | blocked: not reached in the run's time |
| ed10-waveform-matched-controls | blocked: depends on ed2-waveform-validation |

In [ ]:
import json, time, zlib, resource, collections
import numpy as np, pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import huszar as h
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
KEY = "huszar-2022"
def seed(claim): return zlib.crc32(f"{KEY}/{claim}".encode())
SEEDS_PLUS = [0, 1000, 2000, 3000, 4000]
T0 = time.time()

## Data check

In [ ]:
metas = h.survey(workers=8)
S = pd.DataFrame([{"session": m["session_id"], "subject": m["subject_id"], "sex": m["sex"], "age": m["age"],
                   "birthdate": m["birthdate"], "units": m["n_units"],
                   "pyr": sum(c == "Pyramidal Cell" for c in m.get("u_putative_cell_type", [])),
                   "narrow": sum(c == "Narrow Interneuron" for c in m.get("u_putative_cell_type", [])),
                   "wide": sum(c == "Wide Interneuron" for c in m.get("u_putative_cell_type", [])),
                   "unknown_type": sum(c not in ("Pyramidal Cell", "Narrow Interneuron", "Wide Interneuron")
                                       for c in m.get("u_putative_cell_type", [])),
                   "light_resp": int(np.sum(m.get("u_light_responsive", []))),
                   "lfp_min": m["lfp_duration_s"] / 60, "last_spike_min": m["last_spike"] / 60,
                   "lfp_rate": m.get("lfp_rate"), "ripples": len(m["rip_start"]),
                   "states": "/".join(sorted(set(m["sleep"]["label"]))) if m["sleep"] else "",
                   "pulses": m["n_pulses"], "beh_epochs": len(h.behaviour_epochs(m)),
                   "envs": ",".join(e[2] for e in h.behaviour_epochs(m)),
                   "trials": len(m["trials"]["start_time"]) if m["trials"] else 0}
                  for m in metas]).sort_values("session").reset_index(drop=True)
META = {m["session_id"]: m for m in metas}
S["homecage_mouse"] = ~S.subject.isin(h.HOMECAGE_EXCLUDED)
print(len(S), "processed files,", S.subject.nunique(), "subjects,", S.session.nunique(), "distinct session ids")
print("LFP rate(s):", S.lfp_rate.unique(), "Hz; LFP duration against last spike, max |difference| =",
      round(float((S.lfp_min - S.last_spike_min).abs().max()), 2), "min")
print(S.groupby("subject").agg(sex=("sex", "first"), birthdate=("birthdate", "first"), sessions=("session", "size"),
                               units=("units", "sum"), pyr=("pyr", "sum"), light_resp=("light_resp", "sum")).to_string())
print("\ncell types over all files:", S[["pyr", "narrow", "wide", "unknown_type"]].sum().to_dict())
print("sleep-state labels per session:", S.states.value_counts().to_dict())
print("behavioural environments:", collections.Counter(e for s in S.envs for e in s.split(",") if e))
print("ages:", S.age.unique(), " sexes:", S.sex.unique())
# deposited twice? identical spike counts or ripple counts between sessions
dup = S[S.duplicated(["units", "ripples", "pulses"], keep=False)]
print("sessions sharing unit, ripple and pulse counts (possible duplicates):", len(dup))

In [ ]:
# Clock checks (protocol, author step 2): spikes against light pulses, ripples and position, each with a shifted control.
chk_sessions = [s for s in ["e13_16f1_210302", "e14_2m3_201125", "e15_9f1_200720", "e16_3m1_210201", "e15_4m2_191213"]]
CK = pd.DataFrame(h.run_sessions(h.clock_checks, [META[s] for s in chk_sessions]))
print(CK.round(3).to_string())

The files describe 17 mice and 104 sessions, one file per session with no duplicates by unit, ripple and pulse
counts. Every session has units with cell types, light pulses, ripples and LFP at 1,250 Hz; 101 sessions have
wake, non-REM and REM states, one has no REM, and two (e13_1f1_200305 and e13_1f1_200307) have wake only. In
the five sessions checked, light-responsive cells fire 2.7 to 15 times more in the 10 ms after the pulse onsets
than 1 s later, pyramidal cells fire 4.5 to 6.3 times faster in the ripples than in windows shifted by 1 s, and in
the four maze sessions among them the median spatial information falls from between 0.56 and 1.07 bits/s to between
0.13 and 0.19 bits/s when spikes are shifted by 30 s against the position. Spikes, pulses, ripples and position are
therefore consistent with one clock in those sessions; other sessions were not checked this way. The theta
section adds the same contrast for the LFP.

## Counts: cohort, sessions, units

> Claim cohort-birthdates: Chronic recordings were made from 17 adult mice (12 males, 5 females) electroporated
> at E13.5 (n = 4), E14.5 (n = 3), E15.5 (n = 6) and E16.5 (n = 4).

In [ ]:
sub = S.groupby("subject").agg(sex=("sex", "first"), birthdate=("birthdate", "first")).reset_index()
print("mice:", len(sub), " males:", int((sub.sex == "M").sum()), " females:", int((sub.sex == "F").sum()))
print("per birthdate:", sub.birthdate.value_counts().sort_index().to_dict())
st1 = pd.DataFrame({"subject": h.ST1_ORDER})
st1["table_birthdate"] = [h.birthdate(s) for s in h.ST1_ORDER]
st1 = st1.merge(sub, on="subject", how="left")
print("subjects of Supplementary Table 1 found in the dataset:", int(st1.sex.notna().sum()), "of", len(st1),
      "; birthdate agrees:", int((st1.table_birthdate == st1.birthdate).sum()))

**Result: reproduced.** The files hold 17 mice, 12 male and 5 female, with 4, 3, 6 and 4 mice electroporated at E13.5 to E16.5; all 17 animals of Supplementary Table 1 are present with the table's birthdate. Sex and birthdate come from the subject ids, as the files say.

> Claim st1-animal-sessions: Supplementary Table 1 lists each mouse's recording sessions and light-responsive
> cells; the 15 homecage mice contribute 96 sessions and 664 light-responsive cells, and the 17 mice 104 sessions.

In [ ]:
per = S.groupby("subject").agg(sessions=("session", "size"), light_resp=("light_resp", "sum"), pyr=("pyr", "sum"))
T = pd.DataFrame({"subject": h.ST1_ORDER, "table_sessions": h.ST1_SESSIONS, "table_light": h.ST1_LIGHT}).set_index("subject")
T = T.join(per)
T["pct_light"] = (100 * T.light_resp / T.pyr).round(2)
T["sessions_match"] = T.sessions == T.table_sessions
T["light_match"] = T.light_resp == T.table_light
print(T.to_string())
hm = T.index[~T.index.isin(h.HOMECAGE_EXCLUDED)]
print("\nsessions of the 15 homecage mice:", int(T.loc[hm, "sessions"].sum()), "; of all 17:", int(T.sessions.sum()))
print("light-responsive cells of the 15 homecage mice:", int(T.loc[hm, "light_resp"].sum()))
print("mice whose session count matches:", int(T.sessions_match.sum()), "of 17; light-responsive count matches:",
      int(T.light_match.sum()), "of 17")

**Result: reproduced.** Every one of the 17 mice has the table's session count and the table's number of light-responsive cells; the 15 homecage mice give 96 sessions and 664 light-responsive cells, and the 17 mice 104 sessions.

> Claim sessions-durations: The recordings comprise 92 sessions lasting 113.7 to 475.6 min (median 309.1 min),
> each including a homecage period of sleep and wake.

In [ ]:
S["duration_min"] = S.lfp_min
S["has_nrem"] = S.states.str.contains("Non-REM")
for lab, sel in [("96 sessions of the 15 homecage mice", S.homecage_mouse), ("104 sessions of all 17 mice", S.index >= 0)]:
    d = S.loc[sel, "duration_min"]
    print(f"{lab}: n = {len(d)}, median {d.median():.1f} min, range {d.min():.1f} to {d.max():.1f} min; "
          f"all within 102.3-523.2: {bool(((d >= 102.33) & (d <= 523.16)).all())}; "
          f"sessions without non-REM: {int((~S.loc[sel, 'has_nrem']).sum())}")
print("sessions without non-REM sleep:", S.loc[~S.has_nrem, ["session", "states"]].values.tolist())
print("count reading of 92 sessions: none of the session sets has 92; 96 is within 92 +- 9.2:", abs(96 - 92) <= 9.2,
      "; 104 is:", abs(104 - 92) <= 9.2)

**Result: not reproduced.** No session set has 92 sessions; under the 96-session reading the count (96) is within 92 +- 9.2, the median is 309.9 min and every duration lies within 113.7 to 475.6 min, the paper's exact range. Two sessions (e13_1f1_200305, e13_1f1_200307) hold only wake in their sleep states, so the criterion's requirement that every session holds non-REM sleep fails, and the criterion has no partial outcome for this case.

> Claim unit-count: The dataset includes 9,471 well-isolated units, 7,411 putative pyramidal cells and 2,060
> putative interneurons (1,556 narrow waveform, 504 wide waveform).

In [ ]:
for lab, sel in [("(a) 96 sessions of the 15 homecage mice", S.homecage_mouse), ("(b) 104 sessions of all 17 mice", S.index >= 0)]:
    x = S.loc[sel]
    print(f"{lab}: units {x.units.sum()}, pyramidal {x.pyr.sum()}, interneurons {x.narrow.sum() + x.wide.sum()} "
          f"(narrow {x.narrow.sum()}, wide {x.wide.sum()}), other type {x.unknown_type.sum()}")

**Result: reproduced.** Over the 96 sessions of the 15 homecage mice there are 7,411 pyramidal cells and 2,060 interneurons (1,556 narrow, 504 wide), the paper's numbers exactly; the total is 9,472 units, the paper's 9,471 plus one unit without a cell type.

> Claim ed2-light-responsive-count: 664 of 7,411 putative pyramidal neurons (8.96%) were light-responsive, that
> is, birthdated, leaving 6,747 nonresponsive.

In [ ]:
x = S[S.homecage_mouse]
U = pd.DataFrame([{"session": m["session_id"], "subject": m["subject_id"], "ct": c, "lr": l, "p_salt": p, "h_rel": r}
                  for m in metas for c, l, p, r in zip(m["u_putative_cell_type"], m["u_light_responsive"],
                                                       m["u_opto_p_salt"], m["u_opto_h_reliable"])])
U = U[~U.subject.isin(h.HOMECAGE_EXCLUDED)]
pyrU = U[U.ct == "Pyramidal Cell"]
a = int(pyrU.lr.sum()); b = int(((pyrU.p_salt <= 0.001) & (pyrU.h_rel.astype(bool))).sum())
b2 = int(((pyrU.p_salt < 0.001) & (pyrU.h_rel.astype(bool))).sum())
print(f"(a) stored light_responsive: {a} of {len(pyrU)} pyramidal ({100*a/len(pyrU):.2f}%), nonresponsive {len(pyrU)-a}")
print(f"(b) from the stored SALT p and reliability flag (p_salt <= 0.001 and h_reliable): {b}; with p < 0.001 (the code's): {b2}")
print("light_responsive outside pyramidal cells:", int(U[U.ct != 'Pyramidal Cell'].lr.sum()))
print("p_salt missing among pyramidal cells:", int(pyrU.p_salt.isna().sum()))

**Result: reproduced.** The stored labels give 664 light-responsive cells of 7,411 pyramidal cells (8.96%), and the same 664 follow from the stored SALT p and reliability flag.

> Claim spwr-duration: The median duration of SPW-Rs across all recordings is 36.4 ms, under detection limits of
> 20 to 200 ms.

In [ ]:
rows = []
for m in metas:
    d = np.asarray(m["rip_dur"]) * 1000
    hc = h.homecage_intervals(m)
    rip = np.c_[m["rip_start"], m["rip_stop"]]
    k = h.in_intervals(rip[:, 0], hc) & h.in_intervals(rip[:, 1], hc)
    rows.append({"session": m["session_id"], "subject": m["subject_id"], "all": d, "hc": d[k]})
R = pd.DataFrame(rows)
for setlab, sel in [("96 homecage-mouse sessions", ~R.subject.isin(h.HOMECAGE_EXCLUDED)), ("104 sessions", R.index >= 0)]:
    for rl in ["all", "hc"]:
        arrs = R.loc[sel, rl]
        pooled = np.concatenate(arrs.values)
        meds = np.array([np.median(a) for a in arrs if len(a)])
        print(f"{setlab}, ripples={'(i) all detected' if rl=='all' else '(ii) homecage'}: n = {len(pooled)}; "
              f"(a) pooled median {np.median(pooled):.2f} ms; (b) median of session medians {np.median(meds):.2f}; "
              f"(c) mean of session medians {np.mean(meds):.2f}; range {pooled.min():.1f} to {pooled.max():.1f} ms; "
              f"outside 20-200 ms: {int(((pooled < 20) | (pooled > 200)).sum())}")
allr = np.concatenate(R["all"].values)
print("ripple durations: start-stop agree with ripple_durations column to", 
      np.max(np.abs(np.concatenate([(np.asarray(m['rip_stop']) - np.asarray(m['rip_start'])) * 1000 - np.asarray(m['rip_dur']) * 1000 for m in metas]))), "ms")

**Result: reproduced.** Every combination of reading gives a median between 36.00 and 36.80 ms, within 36.4 +- 3.64 ms; the mean of session medians of all detected ripples (the code's reading) is 36.28 ms over the 96 sessions. All 494,578 ripples last 20.0 to 199.2 ms.

> Claim fig1e-labelled-fraction: The fraction of light-responsive pyramidal cells per session follows a
> bell-shaped wave over birthdates, peaking at E15.5 (E13.5 5.39%, E14.5 11.16%, E15.5 14.74%, E16.5 9.16%).

In [ ]:
x = S[S.homecage_mouse].copy()
x["pct"] = 100 * x.light_resp / x.pyr
g = x.groupby("birthdate").pct
print(pd.DataFrame({"sessions": g.size(), "mean": g.mean().round(2), "sd": g.std().round(2)}).to_string())
groups = [x[x.birthdate == b].pct.values for b in h.BIRTHDATES]
for k in SEEDS_PLUS:
    F, p = h.boot_anova(groups, rng=np.random.default_rng(seed("fig1e-labelled-fraction") + k))
    print(f"seed +{k}: bootstrap ANOVA F = {F:.3f}, p = {p:.4f}")
print("by mouse:", x.groupby(["birthdate", "subject"]).pct.mean().round(2).to_dict())

**Result: reproduced.** The per-session means are 5.39, 11.16, 14.74 and 9.16% over 30, 24, 24 and 18 sessions, the paper's values, with F = 8.525 as printed; the bootstrap ANOVA gives p < 0.0002 under all five seeds, E13.5 is lowest and E15.5 highest.

## Homecage single-cell and SPW-R measures

One pass over the 96 sessions of the 15 homecage mice (`h.homecage_session`). The homecage period is the
recording up to the second behavioural subsession, without the light pulse periods and the maze epochs
(claims, opening comment; getHomecageMetrics.m). SPW-Rs are the dataset's (bz_FindRipples, as the Methods),
those starting and ending in the homecage period.

In [ ]:
HM = [META[s] for s in S.loc[S.homecage_mouse, "session"]]
t = time.time()
hc_res = h.run_sessions(h.homecage_session, HM)
CELLS = pd.DataFrame([c for r in hc_res for c in r[0]])
PAIRS = pd.DataFrame([p for r in hc_res for p in r[1]])
HCI = pd.DataFrame([r[2] for r in hc_res])
LR = CELLS[CELLS.lr]
print(f"{time.time()-t:.0f} s; pyramidal cells {len(CELLS)}, light-responsive {len(LR)}; pairs SBD {int(PAIRS.sbd.sum())}, DBD {int((~PAIRS.sbd).sum())}")
print("homecage period per session (min): median", round(HCI.hc_dur.median() / 60, 1), "range", round(HCI.hc_dur.min() / 60, 1), "-", round(HCI.hc_dur.max() / 60, 1))
print("homecage ripples per session: median", int(HCI.n_rip_hc.median()), "range", HCI.n_rip_hc.min(), "-", HCI.n_rip_hc.max(), "; of all ripples", round(HCI.n_rip_hc.sum() / HCI.n_rip.sum(), 3))
print("light-responsive cells per birthdate:", LR.groupby("birthdate").size().to_dict())

In [ ]:
def by_bd(df, col, claim, test="kw", n_seeds=5):
    groups = [df[df.birthdate == b][col].dropna().values for b in h.BIRTHDATES]
    med = [float(np.median(g)) for g in groups]
    print(f"{col}: n = {[len(g) for g in groups]}, medians = {np.round(med, 4).tolist()}, ranges = "
          f"{[(round(float(g.min()), 3), round(float(g.max()), 3)) for g in groups]}")
    ps = []
    for k in SEEDS_PLUS[:n_seeds]:
        f = h.boot_kw if test == "kw" else h.boot_anova
        st, p = f(groups, rng=np.random.default_rng(seed(claim) + k))
        ps.append(p)
    print(f"  bootstrap {'Kruskal-Wallis H' if test=='kw' else 'ANOVA F'} = {st:.3f}; p under the claim's seed and +1000..+4000: {ps}")
    return med, ps

> Claim fig1g-rate-by-birthdate: Earlier-born light-responsive pyramidal neurons have higher firing rates
> (medians 1.57, 1.63, 1.62 and 1.16 Hz for E13.5 to E16.5).

In [ ]:
med, ps = by_bd(LR, "rate", "fig1g-rate-by-birthdate")
print("E16.5 lowest:", int(np.argmin(med)) == 3)
print("within tolerance:", [abs(a - b) <= t for a, b, t in zip(med, [1.57, 1.63, 1.62, 1.16], [0.157, 0.163, 0.162, 0.116])])
print("by mouse (median rate):", LR.groupby(["birthdate", "subject"]).rate.median().round(2).to_dict())

**Result: reproduced.** Medians 1.52, 1.62, 1.54 and 1.15 Hz over 184, 132, 233 and 115 light-responsive cells are within tolerance, E16.5 is lowest, and the bootstrap Kruskal-Wallis gives p = 0.034 (0.034 to 0.041 over the five seeds).

> Claim fig1h-burst-by-birthdate: Burst index is an inverted-U function of birthdate (medians 0.1167, 0.1272,
> 0.136, 0.1082 for E13.5 to E16.5).

In [ ]:
med, ps = by_bd(LR, "burst", "fig1h-burst-by-birthdate")
print("legend 2-5 ms: inverted-U:", h.inverted_u(med), "; within tolerance:",
      [abs(a - b) <= t for a, b, t in zip(med, [0.1167, 0.1272, 0.136, 0.1082], [0.0117, 0.0127, 0.0136, 0.0108])])
med35, ps35 = by_bd(LR, "burst35", "fig1h-burst-by-birthdate", n_seeds=1)
print("code 3-5 ms variant: inverted-U:", h.inverted_u(med35))
print("by mouse (median 2-5 ms):", LR.groupby(["birthdate", "subject"]).burst.median().round(3).to_dict())

**Result: partially reproduced.** The bootstrap Kruskal-Wallis gives p <= 0.0004 under all seeds and the medians form an inverted U (0.088, 0.092, 0.104, 0.080, highest at E15.5), but all four medians are about 25% below the paper's (0.117 to 0.136) and outside tolerance; the code's 3-5 ms variant is lower still (0.074 to 0.095).

> Claim fig2f-ripple-firing-by-birthdate: SPW-R firing rates (medians 7.8, 8.75, 8.01, 6.13 Hz) and
> participation (0.245, 0.268, 0.235, 0.194) of light-responsive neurons are inverted-U functions of birthdate.

In [ ]:
m1, p1 = by_bd(LR, "rip_rate", "fig2f-ripple-firing-by-birthdate")
m2, p2 = by_bd(LR, "rip_part", "fig2f-ripple-firing-by-birthdate")
print("rate: inverted-U", h.inverted_u(m1), "within tolerance", [abs(a - b) <= 0.1 * b for a, b in zip(m1, [7.8, 8.75, 8.01, 6.13])])
print("participation: inverted-U", h.inverted_u(m2), "within tolerance", [abs(a - b) <= 0.1 * b + 1e-12 for a, b in zip(m2, [0.245, 0.268, 0.235, 0.194])])
print("by mouse (median SPW-R rate):", LR.groupby(["birthdate", "subject"]).rip_rate.median().round(2).to_dict())

**Result: reproduced.** SPW-R rate medians 7.48, 8.82, 7.91 and 6.13 Hz and participation 0.237, 0.269, 0.236 and 0.194 are within tolerance, both are inverted U, and both bootstrap Kruskal-Wallis tests give p <= 0.001 under all seeds.

> Claim fig2g-ripple-cofiring: SBD pairs cofire more than DBD pairs in SPW-Rs (median 0.0399 against 0.0231).

In [ ]:
def sbd_dbd(df, col):
    s, d = df[df.sbd][col].dropna(), df[~df.sbd][col].dropna()
    p = stats.ranksums(s, d).pvalue
    print(f"{col}: SBD median {s.median():.4f} (n = {len(s)}), DBD median {d.median():.4f} (n = {len(d)}), "
          f"two-sided Wilcoxon rank-sum p = {p:.3g}; dropped (NaN) SBD {int(df[df.sbd][col].isna().sum())}, DBD {int(df[~df.sbd][col].isna().sum())}")
    return s.median(), d.median(), p
s, d, p = sbd_dbd(PAIRS, "rip_corr")
print("within tolerance:", abs(s - 0.0399) <= 0.004, abs(d - 0.0231) <= 0.0023)
print("per birthdate SBD/DBD medians:", PAIRS.groupby(["birthdate", "sbd"]).rip_corr.median().round(4).to_dict())

**Result: reproduced.** SBD median 0.0385 against DBD 0.0226 (3,738 and 50,171 pairs with a correlation), rank-sum p = 4.7e-98; both within tolerance.

> Claim fig2b-synchrony-per-animal: Same-birthdate (SBD) pairs fire more synchronously than different-birthdate
> (DBD) pairs in 11 of 13 animals.

In [ ]:
rows = []
for sbj, g in PAIRS.groupby("subject"):
    s, d = g[g.sbd], g[~g.sbd]
    ra = (s.rip_corr.dropna(), d.rip_corr.dropna())
    rb = (s["sync"].dropna(), d["sync"].dropna())
    pa = stats.ranksums(*ra).pvalue if len(ra[0]) and len(ra[1]) else np.nan
    rows.append({"subject": sbj, "sbd_pairs": len(s), "dbd_pairs": len(d),
                 "a_sbd_med": ra[0].median() if len(ra[0]) else np.nan, "a_dbd_med": ra[1].median(), "a_p": pa,
                 "a_higher_sig": bool(len(ra[0]) and ra[0].median() > ra[1].median() and pa < 0.05),
                 "b_diff": (rb[0].mean() - rb[1].mean()) if len(rb[0]) else np.nan})
A = pd.DataFrame(rows)
A["b_positive"] = A.b_diff > 0
print(A.round(4).to_string())
el = A[A.sbd_pairs > 0]
t13 = A[~A.subject.isin(["e13_1f1", "e13_1m1"])]
print(f"the 13 animals of the claim (without e13_1f1 and e13_1m1, the two E13.5 mice with one light-responsive cell in most sessions): "
      f"(a) SBD higher with p < 0.05 in {int(t13.a_higher_sig.sum())}; (b) z-scored CCG difference > 0 in {int(t13.b_positive.sum())}")
print(f"animals with SBD pairs: {len(el)}; (a) SBD higher with p < 0.05: {int(el.a_higher_sig.sum())}; "
      f"(a') SBD median higher regardless of p: {int((el.a_sbd_med > el.a_dbd_med).sum())}; (b) mean z-scored CCG difference > 0: {int(el.b_positive.sum())}")

**Result: reproduced.** Of the 13 animals of the claim, SBD pairs cofire more in SPW-Rs than DBD pairs with p < 0.05 in 10 (reading a), and the z-scored CCG of SBD pairs exceeds that of DBD pairs in 12 (reading b); both are in the range 10 to 12. The two E13.5 mice the paper leaves out do have SBD pairs in the files (1 and 2 pairs).

## Theta

One LFP channel per session is read in full (`h.theta_session`), over the 96 homecage-mouse sessions and the
8 sessions of the two novel-only mice. The channel is chosen by the Methods' rule, a channel with a positive
sharp wave above the centre of the pyramidal layer; how one is picked among those that qualify is not said, so
the helper takes the one with the largest ripple power, on the shank of the pyramidal layer centre when that
shank has one and on any shank otherwise (assumption, `pick_theta_channel`); where it was found is counted below.

In [ ]:
TH_SESS = [META[s] for s in S.loc[S.homecage_mouse | S.subject.isin(h.NOVEL_MICE), "session"]]
t = time.time()
th_res = h.run_sessions(h.theta_session_novel, TH_SESS)
TC = pd.DataFrame([c for r in th_res for c in r[0]])
TP = pd.DataFrame([p for r in th_res for p in r[1]])
TO = pd.DataFrame([{k: v for k, v in r[2].items() if k != "theta_corr_mat"} for r in th_res])
TR = pd.DataFrame([x for r in th_res for x in r[3]])
TMAT = {r[2]["session"]: r[2]["theta_corr_mat"] for r in th_res}
print(f"{time.time()-t:.0f} s, {len(th_res)} sessions; peak memory of this process {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.2f} GB")
print("theta channel with a positive sharp wave, by where it was found:", TO["where"].value_counts().to_dict())
print("sharp wave at the theta channel (uV): median", round(TO.sw_theta.median(), 1), "; at the pyramidal layer channel:", round(TO.sw_pyr.median(), 1))
print("theta cycles in homecage wake per session: median", int(TO.n_cycles_hc_wake.median()), "range", TO.n_cycles_hc_wake.min(), "-", TO.n_cycles_hc_wake.max())
hcm = ~TC.subject.isin(h.HOMECAGE_EXCLUDED)
print("clock check, light-responsive cells of homecage mice locked in wake at p < 0.01: phases as given",
      int((TC[hcm].wake_p < 0.01).sum()), "of", int(hcm.sum()), "; with spikes against the phase 10.3 s later",
      int((TC[hcm].wake_p_shift < 0.01).sum()), "; median wake MRL", round(TC[hcm].wake_mrl.median(), 4), "against", round(TC[hcm].wake_mrl_shift.median(), 4))

> Claim fig1f-rem-shift: Earlier-born light-responsive neurons more often shift their preferred theta phase in REM
> sleep (30.89% at E13.5 to 19.7% at E16.5; regression slope -3.42% per day).

In [ ]:
TCH = TC[hcm].copy()
def rem_shift(df, thr):
    b = df[(df.wake_p < thr) & (df.rem_p < thr)].copy()
    wd, rd = np.degrees(b.wake_phase), np.degrees(b.rem_phase)
    b["shift"] = ((wd >= 120) & (wd <= 300)) & ~((rd >= 120) & (rd <= 300))
    return b
def slope_test(b, claim, k):
    rng = np.random.default_rng(seed(claim) + k)
    x = np.array([h.BD_DAY[d] for d in h.BIRTHDATES])
    grp = [b[b.birthdate == d]["shift"].values.astype(float) for d in h.BIRTHDATES]
    fr = np.array([g.mean() * 100 for g in grp])
    sl = np.polyfit(x, fr, 1)[0]
    boots = [np.polyfit(x, [rng.choice(g, len(g)).mean() * 100 for g in grp], 1)[0] for _ in range(1000)]
    pool = np.concatenate(grp)
    null = [np.polyfit(x, [rng.choice(pool, len(g)).mean() * 100 for g in grp], 1)[0] for _ in range(1000)]
    return fr, sl, np.percentile(boots, [2.5, 97.5]), float(np.mean(np.array(null) <= sl))
for thr in (0.01, 0.05):
    b = rem_shift(TCH, thr)
    print(f"Rayleigh threshold {thr}: cells locked in both states per birthdate {b.groupby('birthdate').size().tolist()}")
    for k in (SEEDS_PLUS if thr == 0.01 else [0]):
        fr, sl, ci, p = slope_test(b, "fig1f-rem-shift", k)
        print(f"  seed +{k}: % REM shifting {np.round(fr, 2).tolist()}, slope {sl:.2f} %/day, 95% CI {np.round(ci, 2).tolist()}, one-tailed p = {p:.3f}")
b = rem_shift(TCH, 0.01)
print("within tolerance (0.01):", [abs(a - e) <= t for a, e, t in zip(slope_test(b, 'fig1f-rem-shift', 0)[0], [30.89, 24.5, 23.8, 19.7], [3.09, 2.45, 2.38, 1.97])])
print("by mouse, % shifting (0.01):", (b.groupby(["birthdate", "subject"])["shift"].mean() * 100).round(1).to_dict())

**Result: not reproduced, resting on the choice of theta channel.** The fractions of REM-shifting cells are 27.96, 20.99, 19.74 and 21.54% (E14.5 and E15.5 outside tolerance), the slope is -2.05% per day with a 95% interval of about -6.2 to 2.2, and the one-tailed p is 0.157 to 0.174 over the five seeds, so the slope is not significant. With the code's Rayleigh threshold of 0.05 the slope is -1.98 and p = 0.162.

> Claim fig2d-theta-depth: Intermediate-birthdate neurons have lower theta depth modulation in waking (medians
> 0.127, 0.099, 0.094, 0.103) but not in REM sleep.

In [ ]:
w = TCH[TCH.wake_p < 0.01].rename(columns={"wake_mrl": "wake_depth"})
r = TCH[TCH.rem_p < 0.01].rename(columns={"rem_mrl": "rem_depth"})
mw, pw = by_bd(w, "wake_depth", "fig2d-theta-depth")
mr, pr = by_bd(r, "rem_depth", "fig2d-theta-depth")
print("wake: E14.5 and E15.5 both below E13.5 and E16.5:", max(mw[1], mw[2]) < min(mw[0], mw[3]),
      "; within tolerance:", [abs(a - e) <= 0.1 * e for a, e in zip(mw, [0.127, 0.099, 0.094, 0.103])])
print("by mouse (median wake depth):", w.groupby(["birthdate", "subject"]).wake_depth.median().round(3).to_dict())

**Result: reproduced, resting on the choice of theta channel.** Wake depth modulation medians 0.120, 0.097, 0.090 and 0.107 are within tolerance, E14.5 and E15.5 are below E13.5 and E16.5, and the bootstrap Kruskal-Wallis gives p < 0.0002; in REM sleep the medians are 0.140 to 0.144 and p = 0.74 to 0.75 over the seeds, so no difference is found, as the paper reports.

> Claim fig2c-theta-cofiring: SBD pairs cofire more than DBD pairs in theta cycles (median 0.0189 against 0.0087).

In [ ]:
TPH = TP[~TP.subject.isin(h.HOMECAGE_EXCLUDED)]
s, d, p = sbd_dbd(TPH, "theta_corr")
print("pairs SBD", int(TPH.sbd.sum()), "DBD", int((~TPH.sbd).sum()), "; within tolerance:", abs(s - 0.0189) <= 0.0019, abs(d - 0.0087) <= 0.0009)
pos = set(TO[TO["where"] == "same shank"].session)
s2, d2, p2 = sbd_dbd(TPH[TPH.session.isin(pos)], "theta_corr")
print("variant, sessions whose theta channel is on the shank of the pyramidal layer centre:", round(s2, 4), round(d2, 4))
print("per birthdate SBD/DBD medians:", TPH.groupby(["birthdate", "sbd"]).theta_corr.median().round(4).to_dict())

**Result: reproduced, resting on the choice of theta channel.** SBD median 0.0180 against DBD 0.0078 over 3,727 and 50,388 pairs with a correlation, rank-sum p = 1.3e-41; both medians are within tolerance, the DBD median at the lower edge (0.0078 to 0.0096). In the 35 sessions whose theta channel lies on the shank of the pyramidal layer centre, the SBD median is 0.0145.

> Claim fig6c-theta-rate-preserved: Individual pyramidal neurons preserve their theta-related firing rates
> between the familiar and novel environments (Spearman rho = 0.7378, n = 2,544 neurons).

In [ ]:
NS = S.loc[S.subject.isin(h.NOVEL_MICE), "session"].tolist()
TRN = TR[TR.session.isin(NS)].copy()
def theta_rates(wake_only):
    x = TRN[TRN.wake_only == wake_only]
    fam = x[x.epoch == 0].set_index(["session", "unit"])
    nov = x[x.epoch > 0].groupby(["session", "unit"])[["spikes", "dur"]].sum()
    df = fam[["rate", "lr"]].rename(columns={"rate": "fam"}).join(nov)
    df["nov"] = df.spikes / df.dur
    return df.dropna(subset=["fam", "nov"])
print("novel sessions:", len(NS), "; novel subsessions:", TRN[TRN.epoch > 0].groupby(["session", "epoch"]).ngroups)
for wo in (True, False):
    df = theta_rates(wo)
    rho, p = stats.spearmanr(df.fam, df.nov)
    print(f"{'Methods, cycles in wake' if wo else 'code variant, no wake restriction'}: n = {len(df)} cells, Spearman rho = {rho:.4f}, p = {p:.3g}")
df = theta_rates(True)
print("per mouse rho:", {s: round(stats.spearmanr(g.fam, g.nov)[0], 3) for s, g in df.reset_index().assign(subject=lambda d: d.session.str[:8]).groupby("subject")})
print("theta cycles per subsession (wake only): median", int(TRN[TRN.wake_only].groupby(['session','epoch']).n_cycles.first().median()),
      "min", int(TRN[TRN.wake_only].groupby(['session','epoch']).n_cycles.first().min()))

**Result: reproduced, resting on the choice of theta channel.** Over the 2,544 pyramidal cells of the 12 novel sessions (the paper's n), theta-related rates in the familiar and novel subsessions correlate with Spearman rho = 0.741, p < 1e-300; the same with or without the wake restriction, and 0.735 to 0.745 in each mouse.

> Claim fig6d-rate-lower-novel: Theta-related firing rates are slightly lower in novel environments than in the
> familiar one (median 0.69 against 0.77 Hz).

In [ ]:
for wo in (True, False):
    df = theta_rates(wo)
    p = stats.ranksums(df.nov, df.fam).pvalue
    pw = stats.wilcoxon(df.nov, df.fam).pvalue
    print(f"{'Methods (wake)' if wo else 'code variant'}: familiar median {df.fam.median():.3f} Hz, novel median {df.nov.median():.3f} Hz, "
          f"n = {len(df)}; rank-sum p = {p:.3g}; paired signed-rank p = {pw:.3g}")

**Result: reproduced, resting on the choice of theta channel.** Median theta-related rate 0.755 Hz in the familiar maze and 0.675 Hz in novel environments over 2,544 cells, rank-sum p = 0.0013; both within tolerance.

## Familiar figure-eight maze

> Claim maze-cohort: Nine mice (E13.5 n = 2, E14.5 n = 2, E15.5 n = 2, E16.5 n = 3) ran a spatial alternation
> task on a familiar figure-eight maze, and three mice explored novel environments.

In [ ]:
fam = S[S.subject.isin(h.FAMILIAR_MICE) & (S.beh_epochs > 0)]
print("familiar-maze mice with figure-eight sessions:", fam.subject.nunique(), fam.groupby("birthdate").subject.nunique().to_dict(),
      "; sessions:", len(fam))
print("first environment of their sessions:", fam.envs.str.split(",").str[0].value_counts().to_dict())
nov = S[S.subject.isin(h.NOVEL_MICE)]
nov_has = nov[nov.envs.str.split(",").str.len() > 1]
print("novel-environment mice with a later subsession:", nov_has.subject.nunique(), sorted(nov_has.subject.unique()),
      "; sessions:", len(nov_has), "of", len(nov))
other = S[~S.subject.isin(h.FAMILIAR_MICE | h.NOVEL_MICE) & (S.beh_epochs > 0)]
print("other mice with behavioural subsessions:", other.subject.unique().tolist(), len(other), "sessions")

**Result: reproduced.** The nine mice of the table's Figure 5 column have 57 sessions with a first subsession on the figure-eight (2, 2, 2 and 3 mice per birthdate), and the three mice of the Figure 6 column have later subsessions in all 12 of their sessions.

In [ ]:
FM = [META[s] for s in fam.session]
t = time.time()
mz = h.run_sessions(h.maze_session, FM)
print(f"{time.time()-t:.0f} s, {len(mz)} sessions, pyramidal cells {sum(len(x['pyr']) for x in mz)}, light-responsive {sum(int(x['lr'].sum()) for x in mz)}")

> Claim maze-task: On the figure-eight maze each trial was 205 cm long, position was tracked at 30 Hz, and
> animals ran at least ten trials along each arm in every session.

In [ ]:
tl = pd.DataFrame([{"session": x["session"], "x0": t["x0"], "x1": t["x1"], "arm": t["arm"]} for x in mz for t in x["trials"]])
tl["extent"] = tl.x1 - tl.x0
print("linearized extent per trial (cm): median", round(tl.extent.median(), 1), "range", round(tl.extent.min(), 1), "-", round(tl.extent.max(), 1),
      "; max position", round(tl.x1.max(), 1))
rates = np.array([1 / x["dt"] for x in mz])
print("tracking rate (1/median interval): median", round(float(np.median(rates)), 2), "Hz, range", round(rates.min(), 2), "-", round(rates.max(), 2))
tc = pd.DataFrame([{"session": x["session"], **x["trial_counts"]} for x in mz])
tc["min_type"] = tc[["left", "right"]].min(axis=1); tc["total"] = tc.left + tc.right
print("sessions:", len(tc), "; with >= 10 of each type:", int((tc.min_type >= 10).sum()), "; with >= 20 trials:", int((tc.total >= 20).sum()))
print(tc[tc.min_type < 10].to_string())

**Result: not reproduced.** Trials span a median 202.8 cm of linearized position (maximum 205.0) and tracking runs at 30.0 Hz, both within tolerance, but only 30 of 57 sessions hold at least ten trials of each type, and 6 sessions hold fewer than 20 trials in all, so neither the full nor the partial condition on trial counts holds.

> Claim novel-explorations: The novel-environment mice made 16 novel environment explorations in six distinct
> environments (two linear tracks, three open fields and one figure-eight maze).

In [ ]:
ex = [(m["session_id"], e[2]) for m in metas if m["subject_id"] in h.NOVEL_MICE for e in h.behaviour_epochs(m)[1:]]
print("novel explorations (subsessions after the first):", len(ex))
print("by environment:", collections.Counter(e for _, e in ex))
print("distinct environments:", len(set(e for _, e in ex)))

**Result: reproduced.** The three novel mice have 16 subsessions after the familiar one, in six environments: two linear tracks, three open fields and one novel figure-eight (tmaze_ipshita).

> Claim fig5b-spatial-info-by-birthdate: Spatial information (medians 0.836, 1.05, 1.35, 0.88 bits/s) and spatial
> coverage (36, 44, 44, 18 cm) on the familiar maze are largest in intermediate-birthdate neurons.

In [ ]:
def cell_maze_measures(x):
    out = []
    M, occ = x["maps"], x["occ"]
    for c in range(M.shape[1]):
        rl, rr = M[0, c], M[1, c]
        cat, ocat = np.r_[rl, rr], np.r_[occ[0], occ[1]]
        def cov(rate, o, use_info):
            ok = np.isfinite(rate) & (o > 0)
            p = o[ok] / o[ok].sum(); r = rate[ok]; mr = np.sum(p * r)
            hi = r > 5
            if use_info:
                with np.errstate(divide="ignore", invalid="ignore"):
                    term = (r / mr) * np.log2(r / mr)
                hi = hi & (term > 1)
            return int(hi.sum())
        out.append({"session": x["session"], "subject": x["subject"], "birthdate": x["birthdate"], "lr": bool(x["lr"][c]),
                    "si_cat": h.skaggs_info(cat, ocat),
                    "si_mean": np.nanmean([h.skaggs_info(rl, occ[0]), h.skaggs_info(rr, occ[1])]),
                    "cov_a_cat": cov(cat, ocat, True), "cov_b_cat": cov(cat, ocat, False),
                    "cov_a_mean": np.mean([cov(rl, occ[0], True), cov(rr, occ[1], True)]),
                    "cov_b_mean": np.mean([cov(rl, occ[0], False), cov(rr, occ[1], False)])})
    return out
MC = pd.DataFrame([r for x in mz for r in cell_maze_measures(x)])
ML = MC[MC.lr]
exp_si, exp_cov = [0.836, 1.05, 1.35, 0.88], [36, 44, 44, 18]
for col in ["si_cat", "si_mean"]:
    med, ps = by_bd(ML, col, "fig5b-spatial-info-by-birthdate", n_seeds=5 if col == "si_cat" else 1)
    print(f"  {col}: inverted-U {h.inverted_u(med)}, within tolerance {[abs(a - e) <= 0.1 * e for a, e in zip(med, exp_si)]}")
for col in ["cov_a_cat", "cov_b_cat", "cov_a_mean", "cov_b_mean"]:
    med, ps = by_bd(ML, col, "fig5b-spatial-info-by-birthdate", n_seeds=5 if col == "cov_a_cat" else 1)
    print(f"  {col}: inverted-U {h.inverted_u(med)}, within tolerance {[abs(a - e) <= 0.1 * e for a, e in zip(med, exp_cov)]}")
print("by mouse (median si_cat):", ML.groupby(["birthdate", "subject"]).si_cat.median().round(3).to_dict())

**Result: partially reproduced.** Both measures differ across birthdates (bootstrap Kruskal-Wallis p <= 0.0008 under all readings and seeds) and both have their highest median at E14.5 or E15.5 with E13.5 and E16.5 below it, as the claim states, but no reading brings the medians within tolerance: spatial information is 0.56, 0.63, 0.92 and 0.62 bits/s (concatenated maps) against 0.84 to 1.35, and coverage 29.5, 31.5, 31 and 12 cm (reading a) against 36, 44, 44 and 18.

> Claim fig5c-sbd-ratemap-overlap: SBD pairs have higher spatial ratemap correlations than DBD pairs on the
> familiar maze (median 0.0152 against -0.0256).

In [ ]:
STEM = slice(74, 205)  # bins 75 to 205, post_stem_inds (claims)
def corr_visited(a, b):
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() < 3 or a[ok].std() == 0 or b[ok].std() == 0:
        return np.nan
    return float(np.corrcoef(a[ok], b[ok])[0, 1])
def corr_code(a, b):
    if not (np.isfinite(a).all() and np.isfinite(b).all()) or a.std() == 0 or b.std() == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])
rows = []
for x in mz:
    M = x["maps"]; V = np.concatenate([M[0][:, STEM], M[1][:, STEM]], axis=1)
    lr = np.where(x["lr"])[0]; nr = np.where(~x["lr"])[0]
    for sbd, pp in ((True, [(a, b) for i, a in enumerate(lr) for b in lr[i + 1:]]), (False, [(a, b) for a in lr for b in nr])):
        for a, b in pp:
            rows.append({"session": x["session"], "subject": x["subject"], "birthdate": x["birthdate"], "sbd": sbd, "a": a, "b": b,
                         "r": corr_visited(V[a], V[b]), "r_code": corr_code(V[a], V[b]),
                         "rL": corr_visited(M[0][a, STEM], M[0][b, STEM]), "rR": corr_visited(M[1][a, STEM], M[1][b, STEM]),
                         "dshank": abs(int(x["shank"][a]) - int(x["shank"][b]))})
MP = pd.DataFrame(rows)
s, d, p = sbd_dbd(MP, "r")
print("within tolerance:", abs(s - 0.0152) <= 0.0015, abs(d + 0.0256) <= 0.0026)
s2, d2, p2 = sbd_dbd(MP, "r_code")
print("by mouse SBD/DBD medians:", MP.groupby(["subject", "sbd"]).r.median().round(4).to_dict())

**Result: partially reproduced.** Over 2,609 SBD and 39,965 DBD pairs (the paper's counts), SBD median 0.0189 against DBD -0.0239, rank-sum p = 5.9e-9; the DBD median is within tolerance and the SBD median is above it (0.0137 to 0.0167). The code's variant gives 0.0187 and -0.0249.

> Claim fig5d-overlap-by-distance: Ratemap overlap of SBD pairs is larger than that of DBD pairs within 250
> micrometres (same and adjacent shanks) but not at 500 micrometres.

In [ ]:
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
D5 = MP[MP.dshank <= 2].dropna(subset=["r"]).copy()
D5["dist"] = D5.dshank * 250; D5["grp"] = np.where(D5.sbd, "SBD", "DBD")
print("pairs by shank distance (um) and group:", D5.groupby(["dist", "grp"]).size().to_dict(), "; dropped at 750 um:", int((MP.dshank > 2).sum()))
fit = smf.ols("r ~ C(grp, Sum) * C(dist, Sum)", data=D5).fit()
print(sm.stats.anova_lm(fit, typ=3).round(4).to_string())
means = D5.groupby(["dist", "grp"]).r.mean()
print("means:", means.round(4).to_dict())
expd = {(0, "SBD"): 0.0707, (0, "DBD"): 0.0284, (250, "SBD"): 0.0445, (250, "DBD"): 0.0173, (500, "SBD"): 0.0195, (500, "DBD"): 0.0151}
print("within tolerance:", {k: abs(means[k] - v) <= 0.1 * v for k, v in expd.items()})
tk = pairwise_tukeyhsd(D5.r, D5.grp + "_" + D5.dist.astype(str))
tt = pd.DataFrame(tk.summary().data[1:], columns=tk.summary().data[0])
print(tt[[a.split("_")[1] == b.split("_")[1] for a, b in zip(tt.group1, tt.group2)]].to_string())

**Result: partially reproduced.** The birthdate main effect has p = 0.0001 over 2,561 SBD and 38,497 DBD pairs, and SBD exceeds DBD by Tukey's test at 0 um (p = 0.0002) and 250 um (p = 0.041) but not at 500 um (p = 0.96), as the paper says; five of the six means are outside the tight tolerances (for example SBD 0 um 0.060 against 0.0707).

> Claim fig5e-pv-decorrelation: Population firing-rate vectors at the same linearized position decorrelate between
> left and right trials once the animal leaves the common stem (median correlations stem 0.93, turn 1 0.412,
> arm 0.372, turn 2 0.394).

In [ ]:
SEG = {"stem": (0, 74), "turn 1": (74, 111), "arm": (111, 185), "turn 2": (185, 205)}
pv = []
for x in mz:
    M = x["maps"]
    for b in range(205):
        a1, a2 = M[0][:, b], M[1][:, b]
        ok = np.isfinite(a1) & np.isfinite(a2)
        r = np.corrcoef(a1[ok], a2[ok])[0, 1] if ok.sum() > 2 and a1[ok].std() > 0 and a2[ok].std() > 0 else np.nan
        seg = [k for k, (lo, hi) in SEG.items() if lo <= b < hi][0]
        pv.append({"session": x["session"], "bin": b, "seg": seg, "r": r})
PV = pd.DataFrame(pv)
print("PV correlations per segment:", PV.groupby("seg").r.agg(["median", "count"]).round(4).loc[list(SEG)].to_dict(), "; NaN:", int(PV.r.isna().sum()))
groups = [PV[PV.seg == k].r.dropna().values for k in SEG]
for k in SEEDS_PLUS:
    H_, p = h.boot_kw(groups, rng=np.random.default_rng(seed("fig5e-pv-decorrelation") + k), n_boot=5000 if k == 0 else 1000)
    print(f"seed +{k}: bootstrap Kruskal-Wallis H = {H_:.1f}, p = {p:.4f}")
meds = [np.median(g) for g in groups]
print("within tolerance:", [abs(a - e) <= 0.1 * e for a, e in zip(meds, [0.93, 0.412, 0.372, 0.3944])], "; stem highest:", int(np.argmax(meds)) == 0)

**Result: partially reproduced.** Median PV correlations are 0.930 on the stem, 0.488 on turn 1, 0.407 on the arm and 0.407 on turn 2; the stem is highest and the bootstrap Kruskal-Wallis gives p < 0.0002 under all seeds, but the turn 1 median is outside tolerance (0.371 to 0.453).

> Claim fig5f-remap-together: Between left and right trials, ratemaps of SBD pairs reorganize closer together than
> those of DBD pairs.

In [ ]:
LRp = MP.dropna(subset=["rL", "rR"])
obs = pd.concat([LRp.assign(x=LRp.rL, y=LRp.rR), LRp.assign(x=LRp.rR, y=LRp.rL)])
sb = obs[obs.sbd]
edges = np.percentile(np.r_[sb.rL, sb.rR], [0, 33, 66, 99])
print("tercile edges (SBD, both trial types):", np.round(edges, 4).tolist())
for lab, open_ in (("edges as the code has them", False), ("variant, outer edges open", True)):
    e = edges.copy()
    if open_: e[0], e[-1] = -np.inf, np.inf
    o = obs.copy(); o["ter"] = pd.cut(o.x, e, labels=[1, 2, 3], include_lowest=True)
    o = o.dropna(subset=["ter"]); o["grp"] = np.where(o.sbd, "SBD", "DBD")
    fit = smf.ols("y ~ C(grp, Sum) * C(ter, Sum)", data=o).fit()
    an = sm.stats.anova_lm(fit, typ=3)
    mt = o.groupby(["ter", "grp"], observed=True).y.mean().unstack()
    print(f"{lab}: n SBD {int(o.sbd.sum())}, DBD {int((~o.sbd).sum())}; birthdate F = {an.loc['C(grp, Sum)', 'F']:.2f}, p = {an.loc['C(grp, Sum)', 'PR(>F)']:.3g}; "
          f"tercile p = {an.loc['C(ter, Sum)', 'PR(>F)']:.3g}; interaction p = {an.loc['C(grp, Sum):C(ter, Sum)', 'PR(>F)']:.3g}")
    print("   tercile means:", mt.round(4).to_dict(), "; SBD > DBD in each tercile:", bool((mt.SBD > mt.DBD).all()))

**Result: reproduced.** The birthdate main effect has F = 24.0, p = 9.5e-7 with SBD higher, and SBD exceeds DBD in each tercile (0.020, 0.049, 0.072 against 0.002, 0.022, 0.043); the variant with open outer edges gives the same.

## Novel environments

Rate maps in the familiar figure-eight and in the later subsessions of the 12 sessions of the three novel mice
(`h.novel_session`). Open fields are left out: their positions are in the lab's units, with no scale given, so
the Methods' 1 cm bins cannot be formed. A subsession whose linearized series has another environment's name
or lies outside the epoch is left out too.

In [ ]:
t = time.time()
nv = h.run_sessions(h.novel_session, [META[s] for s in NS])
NE = pd.DataFrame([{"session": x["session"], "epoch": e["epoch"], "env": e["env"], "used": e["maps"] is not None,
                    "why": e.get("why", ""), "trials": e.get("trial_counts")} for x in nv for e in x["epochs"]])
print(f"{time.time()-t:.0f} s"); print(NE.to_string())
print("novel subsessions with maps:", int(NE[(NE.epoch > 0) & NE.used].shape[0]), "of", int((NE.epoch > 0).sum()))

In [ ]:
NOVEL_BINS = {"tmaze_ipshita": slice(50, 164)}  # novel figure-eight bins 51-164 (code, claims); linear tracks all bins
nrows, crows = [], []
for x in nv:
    for e in x["epochs"]:
        if e["maps"] is None: continue
        M, occ = e["maps"], e["occ"]
        sl = STEM if e["epoch"] == 0 else NOVEL_BINS.get(e["env"], slice(None))
        V = np.concatenate([M[0][:, sl], M[1][:, sl]], axis=1)
        for c in range(M.shape[1]):
            cat, ocat = np.r_[M[0, c], M[1, c]], np.r_[occ[0], occ[1]]
            ok = np.isfinite(cat) & (ocat > 0)
            r = cat[ok]; mr = r.mean()
            with np.errstate(divide="ignore", invalid="ignore"):
                eq = float(np.nanmean(np.where(r > 0, r * np.log2(r / mr), 0))) if mr > 0 else np.nan
            crows.append({"session": x["session"], "epoch": e["epoch"], "env": e["env"], "unit": c, "lr": bool(x["lr"][c]),
                          "si": h.skaggs_info(cat, ocat), "si_equal": eq})
        lr = np.where(x["lr"])[0]; nr = np.where(~x["lr"])[0]
        for sbd, pp in ((True, [(a, b) for i, a in enumerate(lr) for b in lr[i + 1:]]), (False, [(a, b) for a in lr for b in nr])):
            for a, b in pp:
                nrows.append({"session": x["session"], "epoch": e["epoch"], "env": e["env"], "sbd": sbd, "a": a, "b": b, "r": corr_visited(V[a], V[b])})
NP = pd.DataFrame(nrows); NC = pd.DataFrame(crows)
print("cells x subsessions:", len(NC), "; pairs x subsessions:", len(NP), "; NaN correlations:", int(NP.r.isna().sum()))

> Claim fig6d-info-lower-novel: Spatial information is lower in novel environments than in the familiar one
> (median 0.39 against 0.88 bits/s).

In [ ]:
for col in ("si", "si_equal"):
    f_, n_ = NC[NC.epoch == 0][col].dropna(), NC[NC.epoch > 0][col].dropna()
    p = stats.ranksums(n_, f_).pvalue
    print(f"{col}: familiar median {f_.median():.3f} bits/s (n = {len(f_)}), novel median {n_.median():.3f} (n = {len(n_)}), rank-sum p = {p:.3g}")
print("novel medians by environment:", NC[NC.epoch > 0].groupby("env").si.median().round(3).to_dict())

**Result: partially reproduced.** Spatial information is lower in the novel linear environments than on the familiar maze (median 0.367 against 0.661 bits/s, rank-sum p = 9.9e-36); the novel median is within tolerance and the familiar one is not (0.792 to 0.968). The open fields, 7 of the 16 novel explorations, are not included.

> Claim fig6c-novel-ratemap-corr-higher: Pairwise spatial ratemap correlations are higher in novel environments
> (median 0.048) than in the familiar environment (median -0.045).

In [ ]:
f_, n_ = NP[NP.epoch == 0].r.dropna(), NP[NP.epoch > 0].r.dropna()
p = stats.ranksums(n_, f_).pvalue
print(f"familiar median {f_.median():.4f} (n = {len(f_)}), novel median {n_.median():.4f} (n = {len(n_)}), rank-sum p = {p:.3g}")
print("within tolerance:", abs(f_.median() + 0.045) <= 0.0045, abs(n_.median() - 0.048) <= 0.0048)
print("novel medians by environment:", NP[NP.epoch > 0].groupby("env").r.median().round(4).to_dict())

**Result: not reproduced.** In the novel environments that could be mapped (8 of 16 subsessions, no open fields), the median pairwise ratemap correlation is -0.032 against -0.045 on the familiar maze, rank-sum p = 0.75; the familiar median is within tolerance, the novel one (0.048 expected) is not.

> Claim fig6e-sbd-novel-overlap: In novel environments SBD pairs have higher ratemap correlations than DBD pairs
> (median 0.117 against 0.046).

In [ ]:
s, d, p = sbd_dbd(NP[NP.epoch > 0], "r")
print("within tolerance:", abs(s - 0.117) <= 0.0117, abs(d - 0.046) <= 0.0046)
print("by environment SBD/DBD medians:", NP[NP.epoch > 0].groupby(["env", "sbd"]).r.median().round(4).to_dict())

**Result: partially reproduced.** In the mapped novel environments SBD pairs correlate more than DBD pairs (median 0.030 against -0.034 over 931 and 21,442 pairs, rank-sum p = 0.002), but both medians are outside tolerance (0.117 and 0.046 expected).

> Claim fig6f-remap-together-novel: Ratemaps of SBD pairs remain closer together than those of DBD pairs between
> the familiar and novel environments.

In [ ]:
famr = NP[NP.epoch == 0][["session", "a", "b", "sbd", "r"]].rename(columns={"r": "fam"})
novr = NP[NP.epoch > 0][["session", "a", "b", "epoch", "r"]].rename(columns={"r": "nov"})
J = novr.merge(famr, on=["session", "a", "b"]).dropna(subset=["fam", "nov"])
J["grp"] = np.where(J.sbd, "SBD", "DBD")
ea = np.percentile(J.fam, [0, 33, 66, 99])
eb = np.percentile(np.r_[famr.fam.dropna(), novr.nov.dropna()], [0, 33, 66, 99])
for lab, e in (("(a) familiar SBD and DBD correlations", ea), ("(b) code, all three columns", eb)):
    e = e.copy(); e[0], e[-1] = -np.inf, np.inf
    o = J.copy(); o["ter"] = pd.cut(o.fam, e, labels=[1, 2, 3])
    fit = smf.ols("nov ~ C(grp, Sum) * C(ter, Sum)", data=o).fit()
    an = sm.stats.anova_lm(fit, typ=3)
    mt = o.groupby(["ter", "grp"], observed=True).nov.mean().unstack()
    print(f"{lab}: n SBD {int(o.sbd.sum())}, DBD {int((~o.sbd).sum())}; birthdate F = {an.loc['C(grp, Sum)', 'F']:.2f}, p = {an.loc['C(grp, Sum)', 'PR(>F)']:.3g}; "
          f"tercile p = {an.loc['C(ter, Sum)', 'PR(>F)']:.3g}; interaction p = {an.loc['C(grp, Sum):C(ter, Sum)', 'PR(>F)']:.3g}")
    print("   tercile means:", mt.round(4).to_dict(), "; terciles with SBD > DBD:", int((mt.SBD > mt.DBD).sum()),
          "; SBD mean over all", round(o[o.sbd].nov.mean(), 4), "DBD", round(o[~o.sbd].nov.mean(), 4))

**Result: reproduced.** Under both readings of the tercile edges the birthdate main effect has p = 0.016 and 0.013 with SBD higher (mean novel correlation 0.040 against 0.016), and SBD exceeds DBD in all three terciles.

## Monosynaptic connections

Pyramidal-to-interneuron CCGs in 0.8 ms bins from spikes outside the light pulse periods, over the 96
homecage-mouse sessions (`h.mono_session`). The Methods give the test as identical to the reliability test
of the light-responsive detection (hollowed Gaussian baseline, 15 ms s.d., Poisson tests at alpha 0.001
Bonferroni corrected, the peak against the bins before zero lag) and refer to English et al. (2017) for the
rest; the lags tested (0.8 to 2.8 ms, the window of the transmission measure) and the hollow fraction (0.6)
are assumptions.

In [ ]:
t = time.time()
mo = h.run_sessions(h.mono_session, HM)
CON = pd.DataFrame([r for x in mo for r in x["rows"]])
PV_ = np.concatenate([x["pvals"] for x in mo])
npairs = sum(x["conn"].size for x in mo)
print(f"{time.time()-t:.0f} s; possible pyramidal-interneuron pairs {npairs}; tested (CCG with >= 50 counts) {len(PV_)}; connections {int(sum(x['conn'].sum() for x in mo))}")
for a in (0.001, 0.0001, 1e-5, 1e-6):
    print(f"  count at alpha {a} (Bonferroni over 3 bins): {int(((PV_[:, 2] < a / 3) & (PV_[:, 3] < a / 3)).sum())}")

> Claim fig7b-sbd-convergence: SBD pyramidal pairs converge onto the same interneurons more than DBD pairs (median
> convergence 0.2727 against 0.2).

In [ ]:
crow = []
for x in mo:
    c = x["conn"]; lr = np.where(x["lr"])[0]; nr = np.where(~x["lr"])[0]
    for i, a in enumerate(lr):
        for b in lr[i + 1:]: crow.append({"session": x["session"], "subject": x["subject"], "birthdate": x["birthdate"], "sbd": True, "cv": h.convergence(c, a, b)})
        for b in nr: crow.append({"session": x["session"], "subject": x["subject"], "birthdate": x["birthdate"], "sbd": False, "cv": h.convergence(c, a, b)})
CV = pd.DataFrame(crow)
s, d, p = sbd_dbd(CV, "cv")
print("within tolerance:", abs(s - 0.2727) <= 0.0273, abs(d - 0.2) <= 0.02)
print("by birthdate SBD/DBD medians:", CV.groupby(["birthdate", "sbd"]).cv.median().round(4).to_dict())

**Result: partially reproduced, resting on the assumed connection detector.** SBD pairs converge more than DBD pairs (median 0.455 against 0.364, rank-sum p = 1.3e-79), but both medians are far above the paper's (0.273 and 0.200); the detector finds 50,618 connections against the paper's 31,932.

> Claim fig7ce-convergence-by-birthdate: Convergence onto interneurons (DBD-corrected medians 0.0857, 0.1071, 0.1,
> 0.0285) and spike transmission probability (0.0184, 0.0202, 0.0222, 0.0199) are bell-shaped functions of the
> presynaptic neurons' birthdate.

In [ ]:
sb = CV[CV.sbd].dropna().copy()
dm = CV[~CV.sbd].groupby("birthdate").cv.median()
sb["cv_corr"] = sb.cv - sb.birthdate.map(dm)
m1, p1 = by_bd(sb, "cv_corr", "fig7ce-convergence-by-birthdate")
print("corrected convergence: inverted-U", h.inverted_u(m1), "within tolerance", [abs(a - e) <= 0.1 * abs(e) for a, e in zip(m1, [0.0857, 0.1071, 0.1, 0.0285])])
L7 = CON[CON.pre_lr]
m2, p2 = by_bd(L7, "stp", "fig7ce-convergence-by-birthdate")
print("transmission: inverted-U", h.inverted_u(m2),
      "; within tolerance, legend:", [abs(a - e) <= 0.1 * e for a, e in zip(m2, [0.0184, 0.0202, 0.0222, 0.0199])],
      "; table:", [abs(a - e) <= 0.1 * e for a, e in zip(m2, [0.0238, 0.02356, 0.0247, 0.0195])])

**Result: partially reproduced, resting on the assumed connection detector.** Corrected convergence (medians 0.117, 0.143, 0.100, 0.054) and transmission probability (0.0141, 0.0153, 0.0172, 0.0150) are both inverted U across birthdates with bootstrap Kruskal-Wallis p < 0.0002, but only one median of the eight (corrected convergence, E15.5) is within tolerance under either reading of the transmission medians.

> Claim fig7d-transmission-gamma: Spike transmission probability peaks at presynaptic interspike intervals of 15 to
> 23 ms, in the gamma range, with short-term depression at shorter intervals.

In [ ]:
isi = np.array(CON.stp_isi.tolist(), float)
centres = np.sqrt(h.ISI_EDGES[:-1] * h.ISI_EDGES[1:]) * 1000
mean = np.nanmean(isi, 0)
print(pd.DataFrame({"isi_lo_ms": h.ISI_EDGES[:-1] * 1000, "isi_hi_ms": h.ISI_EDGES[1:] * 1000, "centre_ms": centres,
                    "mean_stp": mean, "n": np.isfinite(isi).sum(0)}).round(4).to_string())
k = int(np.nanargmax(mean))
print(f"peak bin {h.ISI_EDGES[k]*1000:.1f}-{h.ISI_EDGES[k+1]*1000:.1f} ms (centre {centres[k]:.1f}); in 13.5-25.3 ms: {13.5 <= centres[k] <= 25.3}; "
      f"shortest bin {mean[0]:.4f} below peak {mean[k]:.4f}: {mean[0] < mean[k]}")
for lab, sel in (("light-responsive presynaptic only", CON.pre_lr.values),):
    mm = np.nanmean(isi[sel], 0); kk = int(np.nanargmax(mm)); print(lab, "peak centre", round(centres[kk], 1), "ms")

**Result: not reproduced, resting on the assumed connection detector.** Mean transmission probability rises from 0.0190 at 5-7 ms ISIs to a peak of 0.0269 in the 29-42 ms bin (centre 34.9 ms), outside 13.5 to 25.3 ms; the shortest-ISI bin is below the peak, as the claim's depression at short intervals says.

> Claim fig7fg-convergence-cofiring: Of 184,230 possible pyramidal-interneuron pairs, 31,932 are monosynaptically
> connected; 65.27% of pyramidal pairs share at least one postsynaptic interneuron, and cofiring in SPW-Rs, theta
> cycles and ratemap overlap all increase with convergence.

In [ ]:
RMAT = {r[2]["session"]: r[2]["rip_corr_mat"] for r in hc_res}
MAPV = {}
for x in mz:
    MAPV[x["session"]] = np.concatenate([x["maps"][0][:, STEM], x["maps"][1][:, STEM]], axis=1)
rows, nz, tot, anyt = [], 0, 0, 0
for x in mo:
    c = x["conn"].astype(int); n = len(c)
    if n < 2: continue
    sh = c @ c.T; un = c.shape[1] - (1 - c) @ (1 - c).T  # shared targets; union of targets
    iu = np.triu_indices(n, 1)
    nz += int((sh[iu] > 0).sum()); tot += len(iu[0]); anyt += int((un[iu] > 0).sum())
    lr = x["lr"]; rm = RMAT[x["session"]]; tm = TMAT[x["session"]]; V = MAPV.get(x["session"])
    for a, b in zip(*iu):
        if sh[a, b] == 0: continue
        rows.append({"session": x["session"], "nr_pair": (not lr[a]) and (not lr[b]), "cv": sh[a, b] / un[a, b],
                     "rip": rm[a, b], "theta": tm[a, b], "map": corr_visited(V[a], V[b]) if V is not None else np.nan})
Q = pd.DataFrame(rows)
nconn = int(sum(x["conn"].sum() for x in mo))
print(f"connections {nconn} of {npairs} possible; pyramidal pairs with >= 1 shared interneuron {nz} of {tot} ({100*nz/tot:.2f}%); "
      f"of pairs with any target {anyt} ({100*nz/max(anyt,1):.2f}%)")
def quint_reg(df, col, claim, k):
    d = df.dropna(subset=[col]).copy()
    d["q"] = pd.qcut(d.cv.rank(method="first"), 5, labels=False) + 1
    X = d.q.values.astype(float); Y = d[col].values
    def tstat(x, y):
        xm = x - x.mean(); b = (xm @ (y - y.mean())) / (xm @ xm)
        res = y - y.mean() - b * xm; se = np.sqrt((res @ res) / (len(y) - 2) / (xm @ xm)); return b, b / se
    b, t = tstat(X, Y)
    rng = np.random.default_rng(seed(claim) + k)
    null = np.array([tstat(X, rng.choice(Y, len(Y)))[1] for _ in range(1000)])
    return b, t, float(np.mean(np.abs(null) >= abs(t))), d.groupby("q")[col].mean().round(4).tolist(), len(d)
for lab, sel in (("(a) nonresponsive pairs", Q.nr_pair), ("(b) all pyramidal pairs", Q.index >= 0)):
    for col in ("rip", "theta", "map"):
        b, t, p, qm, n = quint_reg(Q[sel], col, "fig7fg-convergence-cofiring", 0)
        print(f"{lab}, {col}: n = {n}, quintile means {qm}, slope {b:.4f}, t = {t:.2f}, bootstrap p = {p:.3f}")
for k in SEEDS_PLUS[1:]:
    print(f"seed +{k} (a):", [round(quint_reg(Q[Q.nr_pair], col, 'fig7fg-convergence-cofiring', k)[2], 3) for col in ("rip", "theta", "map")])

**Result: not reproduced, resting on the assumed connection detector.** The possible pairs are 184,230, the paper's number exactly, but the detector finds 50,618 connections (31,932 +- 3,193 expected) and 86.2% of pyramidal pairs share an interneuron (65.27 +- 6.53% expected). SPW-R cofiring, theta cofiring and ratemap correlation all rise with convergence under both readings (slopes 0.0040, 0.0029 and 0.0017 for nonresponsive pairs, bootstrap p < 0.001 under all seeds), so the slope part holds.

## Blocked claims

> Claim ed2-waveform-validation: Light-evoked spikes most resemble the spontaneous spikes of their own cluster, and
> light-responsive and nonresponsive pyramidal cells have the same waveform trough-to-peak.

The waveform distance needs the waveforms of light-evoked spikes, which need the wide-band signal around the
pulses. The processed files hold only each unit's mean waveform over all spikes; the wide-band signal is in the
11 raw files, of which 5 are homecage-mouse sessions (52 to 141 GB each). The claim is blocked. The
trough-to-peak part alone is printed here for the record; it does not decide the outcome.

In [ ]:
rows = []
for m in metas:
    if m["subject_id"] in h.HOMECAGE_EXCLUDED: continue
    for c, l, v in zip(m["u_putative_cell_type"], m["u_light_responsive"], m["u_trough_to_peak"]):
        if c == "Pyramidal Cell": rows.append({"lr": l, "ttp": v})
TT = pd.DataFrame(rows)
a, b = TT[TT.lr].ttp.dropna(), TT[~TT.lr].ttp.dropna()
print(f"trough-to-peak: light-responsive mean {a.mean():.3f} ms (n = {len(a)}), nonresponsive mean {b.mean():.3f} ms (n = {len(b)}), "
      f"two-sided t-test p = {stats.ttest_ind(a, b).pvalue:.3g}")

The other blocked claims are not computed here: the histology of ed1-brdu-specificity and fig1c-radial-depth is
not in the dataset; ed5-matched-controls needs each unit's L-ratio, which the units table does not hold; the
assembly claims (fig3c, fig3e, ed6, ed7) and the model of fig4d were not run (see the attempt record for the
estimate); ed3-depth-control was not reached in the run's time; ed10-waveform-matched-controls depends on
ed2-waveform-validation, which is blocked.

In [ ]:
print(f"notebook runtime {(time.time() - T0)/60:.1f} min; peak memory of this process {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.2f} GB")

## Summary

| Claim | Outcome |
|---|---|
| cohort-birthdates | reproduced |
| st1-animal-sessions | reproduced |
| sessions-durations | not reproduced (two sessions without sleep; all else holds) |
| spwr-duration | reproduced |
| unit-count | reproduced |
| ed2-light-responsive-count | reproduced |
| ed2-waveform-validation | blocked: data missing |
| ed1-brdu-specificity, fig1c-radial-depth | blocked: data missing |
| fig1e-labelled-fraction | reproduced |
| fig1f-rem-shift | not reproduced (assumed theta channel) |
| fig1g-rate-by-birthdate | reproduced |
| fig1h-burst-by-birthdate | partially reproduced |
| ed3-depth-control | blocked: not reached |
| fig2b-synchrony-per-animal | reproduced |
| fig2c-theta-cofiring | reproduced (assumed theta channel) |
| fig2d-theta-depth | reproduced (assumed theta channel) |
| fig2f-ripple-firing-by-birthdate | reproduced |
| fig2g-ripple-cofiring | reproduced |
| ed5-matched-controls | blocked: data missing (L-ratio) |
| fig3c, fig3e, ed6, ed7, fig4d | blocked: too large |
| maze-cohort | reproduced |
| maze-task | not reproduced (trial counts) |
| novel-explorations | reproduced |
| fig5b-spatial-info-by-birthdate | partially reproduced |
| fig5c-sbd-ratemap-overlap | partially reproduced |
| fig5d-overlap-by-distance | partially reproduced |
| fig5e-pv-decorrelation | partially reproduced |
| fig5f-remap-together | reproduced |
| fig6c-theta-rate-preserved | reproduced (assumed theta channel) |
| fig6d-rate-lower-novel | reproduced (assumed theta channel) |
| fig6d-info-lower-novel | partially reproduced |
| fig6c-novel-ratemap-corr-higher | not reproduced |
| fig6e-sbd-novel-overlap | partially reproduced |
| fig6f-remap-together-novel | reproduced |
| fig7b-sbd-convergence | partially reproduced (assumed detector) |
| fig7ce-convergence-by-birthdate | partially reproduced (assumed detector) |
| fig7d-transmission-gamma | not reproduced (assumed detector) |
| fig7fg-convergence-cofiring | not reproduced (assumed detector) |
| ed10-waveform-matched-controls | blocked: data missing (depends on ed2-waveform-validation) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

How the analysis differs from the paper's. The theta channel is chosen by a rule of this attempt that
follows the Methods' description, and in 62 of 104 sessions no channel with a positive sharp wave was found,
so the pyramidal layer channel was used. The monosynaptic detector follows the Methods' description with
assumed lags, kernel and Bonferroni count, and finds about 60% more connections than the paper. Open fields
are left out of the novel-environment analyses for lack of a spatial scale. Rate maps use a constant-velocity
Kalman filter whose settings the paper does not give. The bootstrap post hoc test is replaced by Tukey's test
on the six distance and birthdate groups.

What was found about the dataset. The files give the paper's counts of mice, sessions, units, cell types
and light-responsive cells exactly, and the paper's per-session labelled fractions. Two sessions of e13_1f1 hold
wake only in their sleep states. In e16_3m2_211211 the position series named for the linear_manu
subsession lies inside the openField_thomas subsession. Positions outside the familiar maze have no spatial
scale. The units table has no L-ratio. Mice e13_1f1 and e13_1m1, which the paper says had a single
light-responsive cell per session, have 1 and 2 same-birthdate pairs in the files. 27 of 57 familiar-maze
sessions hold fewer than ten trials of one type.